In [11]:
!pip install -q sentence-transformers numpy

In [12]:
from google.colab import files
import os

uploaded = files.upload()

file_name = list(uploaded.keys())[0]
with open(file_name, "r", encoding="utf-8") as f:
    text = f.read()

print("File loaded:", file_name)
print("Total characters:", len(text))
print("Number of paragraphs:", len([p for p in text.split("\n") if p.strip()]))

Saving World_cities.txt to World_cities (2).txt
File loaded: World_cities (2).txt
Total characters: 2094
Number of paragraphs: 10


In [13]:
CHUNK_SIZE = 400
OVERLAP_PERCENT = 0.15
OVERLAP = int(CHUNK_SIZE * OVERLAP_PERCENT)
STEP = CHUNK_SIZE - OVERLAP

def chunk_text(text, chunk_size, step):
    chunks = []
    for start in range(0, len(text), step):
        chunk = text[start:start + chunk_size].strip()
        if chunk:
            chunks.append(chunk)
        if start + chunk_size >= len(text):
            break
    return chunks

chunks = chunk_text(text, CHUNK_SIZE, STEP)
print("Chunk size:", CHUNK_SIZE, "| Overlap:", OVERLAP, "characters")
print("Number of chunks:", len(chunks))

Chunk size: 400 | Overlap: 60 characters
Number of chunks: 6


In [14]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_embeddings = model.encode(chunks)
print("Embeddings shape:", chunk_embeddings.shape)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings shape: (6, 384)


In [15]:
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def search(query, top_k=3):

    query_embedding = model.encode(query)

    scores = [cosine_similarity(query_embedding, emb) for emb in chunk_embeddings]

    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]

    print("Question:", query)
    print("=" * 80)
    for rank, (idx, score) in enumerate(ranked, start=1):
        print(f"\nAnswer {rank} (Chunk #{idx}) - Similarity Score: {score:.4f}")
        print("-" * 80)
        print(chunks[idx])
    print("\n" + "=" * 80)


query = input("Enter your question: ")
search(query)

Enter your question: what is the largest city based on population ?
Question: what is the largest city based on population ?

Answer 1 (Chunk #0) - Similarity Score: 0.6380
--------------------------------------------------------------------------------
The largest cities in the world are usually measured by urban agglomeration, which counts the whole built-up area instead of only the official city limits. Most of the biggest cities are in Asia, and rankings change as populations grow and statistical methods are updated.

Jakarta, the capital of Indonesia, is ranked by recent United Nations estimates as the most populous urban area in the world,

Answer 2 (Chunk #4) - Similarity Score: 0.5959
--------------------------------------------------------------------------------
d is the largest city in Africa and the Arab world. It lies along the Nile River near the pyramids of Giza.

Mexico City, the capital of Mexico, has about 22 million people and is the largest city in North America by 